In [ ]:
import threading
import time
import random
import datetime
import rclpy

from IPython.display import clear_output
from rclpy.node import Node
from rclpy.executors import MultiThreadedExecutor
from rclpy.action import ActionClient
from std_msgs.msg import String, Header
from geographic_msgs.msg import GeoPoint
from lotusim_msgs.msg import MASCmd as MASCmdMsg, VesselPositionArray
from lotusim_msgs.srv import SetWaypoints
from lotusim_msgs.action import MASCmd, MASCmdArray

The messages (MASCmd,VesselPositionArray) are in the interface folder. Please open it. 

In MASCmd, the message is divided into goal, result, feedback while its running. We are mainly interested in the goal and result msg 
```
std_msgs/Header header
MASCmd cmd
---
bool result
string name
uint16 entity
---
bool fb
```

Each variable is treated like a class variable. To access the MASCmd under goal, use `msg.cmd` too access.

Below are example of publisher, subscriber and action server.

In [ ]:
SPAWN_LATITUDE = 1.2605794416293148
SPAWN_LONGITUDE = 103.7516212463379
SPAWN_ALTITUDE = 0.0
OFFSET = 0.0001


class ExampleNode(Node):
    def __init__(self):
        super().__init__("example_spawn_node", namespace="/lotusim")
        self.pose_subscription = self.create_subscription(
            VesselPositionArray, "poses", self.poses_callback, 10
        )
        self.random_publisher = self.create_publisher(String, "topic", 10)
        self.mas_action_client = ActionClient(self, MASCmd, "mas_cmd")
        self.mas_array_action_client = ActionClient(self, MASCmdArray, "mas_cmd_array")
        self.vessel_poses = {}
        self.spawned_vessels = []  # track spawned vessels for cleanup
        self.waypoint_client = {}
        self.vessel_id = 0  # ID number for the number of model in the simulation. Used in randomizing location spawned

    def poses_callback(self, msg):
        for vessel_position in msg.vessels:
            lat = vessel_position.geo_point.latitude
            long = vessel_position.geo_point.longitude
            self.vessel_poses[vessel_position.vessel_name] = (lat, long)

    def delete_all_vessels(self):
        if not self.spawned_vessels:
            print("No vessels to delete")
            return

        if not self.mas_array_action_client.wait_for_server(timeout_sec=2.0):
            print("Action server not available for cleanup")
            return

        goal_msg = MASCmdArray.Goal()
        for name in self.spawned_vessels:
            msg = MASCmdMsg()
            msg.cmd_type = MASCmdMsg.DELETE_CMD
            msg.vessel_name = name
            goal_msg.cmd.append(msg)

        # send goal and wait —> executor in background thread to processes it
        goal_future = self.mas_array_action_client.send_goal_async(goal_msg)

        goal_handle = goal_future.result()
        if not goal_handle or not goal_handle.accepted:
            print("Delete goal rejected")
            return

        print("All vessels deleted")
        self.spawned_vessels.clear()

    def spawn_ship_with_dynamics(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "lrauv"
        vessel_name = "lrauv_" + str(self.vessel_id)
        msg.vessel_name = vessel_name
        latlong_msg = GeoPoint()
        latlong_msg.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice(
            [-1, 1]
        )
        latlong_msg.longitude = (
            SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        )
        latlong_msg.altitude = -30.0
        msg.geo_point = latlong_msg

        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
            <underwater>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12346</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </underwater>
            <surface>
                <connection_type>XDynWebSocket</connection_type>
                <uri>ws://127.0.0.1:12345</uri>
                <thrusters>
                    <thrusters1>thruster1</thrusters1>
                </thrusters>
            </surface>
            <init_state>Underwater</init_state>
            </physics_engine_interface>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_aerial_drone(self):
        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "x500"
        vessel_name = "x500_" + str(self.vessel_id)
        msg.vessel_name = vessel_name
        latlong_msg = GeoPoint()
        latlong_msg.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice(
            [-1, 1]
        )
        latlong_msg.longitude = (
            SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        )
        latlong_msg.altitude = 20.0
        msg.geo_point = latlong_msg

        msg.sdf_string = """
        <lotus_param>
            <render_interface>
                <publish_render>true</publish_render>
                <renderer_type_name>lrauv</renderer_type_name>
            </render_interface>
            <physics_engine_interface>
                  <aerial>
                    <connection_type>ROS2</connection_type>
                    <namespace>aerialWorld</namespace>
                  </aerial>
                  <init_state>Aerial</init_state>
            </physics_engine_interface>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg = MASCmd.Goal()
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        return self.mas_action_client.send_goal_async(goal_msg)

    def spawn_circling_ship(self):
        goal_msg = MASCmd.Goal()

        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "dtmb_hull"
        vessel_name = "dtmb_" + str(self.vessel_id)
        msg.vessel_name = vessel_name
        latlong_msg = GeoPoint()
        latlong_msg.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice(
            [-1, 1]
        )
        latlong_msg.longitude = (
            SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        )
        latlong_msg.altitude = 0.0
        msg.geo_point = latlong_msg

        msg.sdf_string = """
        <lotus_param>
            <waypoint_follower>
                <follower>
                    <loop>true</loop>
                    <linear_accel_limit>0.5</linear_accel_limit>
                    <angular_accel_limit>0.005</angular_accel_limit>
                    <angular_velocities_limits>0.01</angular_velocities_limits>
                    <range_tolerance>2</range_tolerance>
                    <circle>
                        <radius>10</radius>
                    </circle>
                </follower>
            </waypoint_follower>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        self.mas_action_client.send_goal_async(goal_msg)

    def spawn_power_ship(self):
        goal_msg = MASCmd.Goal()

        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = "commando"
        vessel_name = "commando_" + str(self.vessel_id)
        msg.vessel_name = vessel_name
        latlong_msg = GeoPoint()
        latlong_msg.latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice(
            [-1, 1]
        )
        latlong_msg.longitude = (
            SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        )
        latlong_msg.altitude = 0.0
        msg.geo_point = latlong_msg

        msg.sdf_string = """
        <lotus_param>
        </lotus_param>
        """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        self.mas_action_client.send_goal_async(goal_msg)

    def spawn_model(self, model_name,x=0,y=0):
        goal_msg = MASCmd.Goal()

        msg = MASCmdMsg()
        msg.cmd_type = MASCmdMsg.CREATE_CMD
        msg.model_name = model_name
        vessel_name = model_name + "_" + str(self.vessel_id)
        msg.vessel_name = vessel_name
        msg.vessel_position.position.x = x
        msg.vessel_position.position.y = y
        msg.sdf_string = """
            <lotus_param>
            </lotus_param>
            """
        self.vessel_id += 1
        self.spawned_vessels.append(vessel_name)
        goal_msg.cmd = msg
        self.mas_action_client.wait_for_server()
        self.mas_action_client.send_goal_async(goal_msg)

    def spawn_multiple_circling_ship(self, number_of_ships):
        goal_msg = MASCmdArray.Goal()

        count = 0
        while count < number_of_ships:
            msg = MASCmdMsg()
            msg.cmd_type = MASCmdMsg.CREATE_CMD
            msg.model_name = "dtmb_hull"
            vessel_name = "dtmb" + str(self.vessel_id)
            msg.vessel_name = vessel_name
            count += 1
            latlong_msg = GeoPoint()
            latlong_msg.latitude = (
                SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
            )
            latlong_msg.longitude = (
                SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
            )
            latlong_msg.altitude = 0.0
            msg.geo_point = latlong_msg

            msg.sdf_string = """
            <lotus_param>
                <waypoint_follower>
                    <follower>
                        <loop>true</loop>
                        <linear_accel_limit>0.5</linear_accel_limit>
                        <angular_accel_limit>0.005</angular_accel_limit>
                        <angular_velocities_limits>0.01</angular_velocities_limits>
                        <range_tolerance>2</range_tolerance>
                        <circle>
                            <radius>10</radius>
                        </circle>
                    </follower>
                </waypoint_follower>
            </lotus_param>
            """
            goal_msg.cmd.append(msg)
            self.vessel_id += 1
            self.spawned_vessels.append(vessel_name)
        self.mas_array_action_client.wait_for_server()
        return self.mas_array_action_client.send_goal_async(goal_msg)

    def setupRosForModel(self, model_name: str):
        if not model_name in self.waypoint_client:
            client = self.create_client(SetWaypoints, model_name + "/waypoints")
            for attempt in range(1, 4):
                if client.wait_for_service(timeout_sec=1.0):
                    self.waypoint_client[model_name] = client
                    raise ValueError("service client unable to be created")
                print(f"Waiting for waypoint service... (attempt {attempt}/3)")
            print(f"Failed to connect to waypoint service for {model_name}")

    def send_random_waypoint_request(self, model_name: str):
        if not model_name in self.waypoint_client:
            try:
                self.setupRosForModel(model_name)
            except ValueError as e:
                print("Sending waypoint failed. Try again.")
                return

        request = SetWaypoints.Request()

        request.header = Header()
        request.header.stamp = self.get_clock().now().to_msg()
        request.header.frame_id = "world"

        latitude = SPAWN_LATITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        longitude = SPAWN_LONGITUDE + OFFSET * self.vessel_id * random.choice([-1, 1])
        request.path = [
            GeoPoint(latitude=latitude, longitude=longitude, altitude=SPAWN_ALTITUDE)
        ]

        request.loop = False

        future = self.waypoint_client[model_name].call_async(request)
        while not future.done():
            time.sleep(0.1)
        # rclpy.spin_until_future_complete(self, future)

        response = future.result()
        if response.success:
            print(f"Waypoint set: ({latitude:.7f}, {longitude:.7f})")
        else:
            print("Failed to set waypoints")

Construct the example node and start the node in a parallel thread

In [ ]:
if not rclpy.ok(): 
    rclpy.init(args=None)

try:
    stop_executor()  
except NameError:
    pass 
except Exception as e:
    print(f"Could not destroy resources: {e}")

node = ExampleNode()
executor = MultiThreadedExecutor()
executor.add_node(node)
stop_event = threading.Event()

def spin_executor():
    executor.spin()
def stop_executor():
    """Call this function to stop the spinning thread"""
    try:
        node.delete_all_vessels()   # cleanup before stopping
        time.sleep(1.0)  # time for Gazebo to cleanup
        stop_event.set()
        executor.shutdown()
        spin_thread.join()
        display_thread.join()
        node.destroy_node()
        time.sleep(2)
        print("Executor stopped successfully")
        
    except Exception as e:
        print(f"Error stopping executor: {e}")

def display_loop():
    while not stop_event.is_set():
        clear_output(wait=True)
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}]")
        for name, (lat, lon) in node.vessel_poses.items():
            print(f"  {name}: lat={lat:.6f}, lon={lon:.6f}")        
        time.sleep(1)

spin_thread = threading.Thread(target=spin_executor, daemon=True)
spin_thread.start()
display_thread = threading.Thread(target=display_loop, daemon=True)
display_thread.start()

In [ ]:
# Spawn 1 ship
node.spawn_multiple_circling_ship(1)

In [ ]:
# Sending waypoint to ship with waypoint follower
node.send_random_waypoint_request("dtmb0")

In [ ]:
node.spawn_power_ship()

In [ ]:
node.spawn_model("commando",0.0,0.0)
node.spawn_model("cube", 5.0, 5.0)

In [ ]:
stop_executor()